# Bioinformática — Del ADN a la proteína con Biopython

Notebook con la resolución de los 6 ejercicios: parte **manual/teórica** (celdas Markdown) y **extensión con Biopython** (celdas de código), comprobando siempre que el resultado automático coincide con el manual.

| Ejercicio | Tema |
|---|---|
| 1 | Replicación del ADN (semiconservativa, enzimas) |
| 2 | Transcripción ADN → ARNm (lectura de FASTA) |
| 3 | Traducción ARNm → proteína y mutaciones |
| 4 | Splicing alternativo (+ FGFR2 en Ensembl) |
| 5 | Proteínas: secuencia, estructura y función (+ PDB) |
| 6 | Pipeline integrador del dogma central |

> Las celdas que consultan bases de datos (NCBI, Ensembl, RCSB PDB) necesitan conexión a internet; si fallan, usan un respaldo local para que el notebook se ejecute entero.

In [ ]:
# Si hace falta:  %pip install biopython pandas
import random, json, logging, urllib.request
import pandas as pd
import Bio
from Bio.Seq import Seq
from Bio import SeqIO, Entrez
from Bio.SeqRecord import SeqRecord
from Bio.SeqUtils import seq3
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from Bio.SeqUtils.ProtParamData import kd   # escala de hidropatía Kyte-Doolittle
from Bio.Data import CodonTable

random.seed(42)
print("Biopython", Bio.__version__)

def codones(s):
    s = str(s)
    return " ".join(s[i:i+3] for i in range(0, len(s), 3))

def duplex(sup, inf_3a5, etiqueta_sup="", etiqueta_inf=""):
    # Imprime una doble hebra: superior 5'->3' e inferior 3'->5' alineadas
    print(f"5' - {codones(sup)} - 3'   {etiqueta_sup}")
    print(f"3' - {codones(inf_3a5)} - 5'   {etiqueta_inf}")

---
## Ejercicio 1. Replicación del ADN

```
5' – ATG CCG TTA GCT – 3'
3' – TAC GGC AAT CGA – 5'
```

### Resolución manual
La replicación es **semiconservativa**: las dos hebras se separan y cada una actúa como molde para una hebra nueva complementaria y antiparalela (A–T, G–C). Cada molécula hija conserva una hebra parental y una nueva.

**Molécula hija 1** (molde = hebra superior):
```
5' – ATG CCG TTA GCT – 3'   (parental)
3' – TAC GGC AAT CGA – 5'   (NUEVA)
```
**Molécula hija 2** (molde = hebra inferior):
```
5' – ATG CCG TTA GCT – 3'   (NUEVA)
3' – TAC GGC AAT CGA – 5'   (parental)
```
Las dos moléculas son idénticas a la original. Escrita 5'→3', la hebra nueva de la hija 1 es `5'-AGC TAA CGG CAT-3'` (complementaria inversa).

### Enzimas
| Enzima | Función |
|---|---|
| **Helicasa** | Rompe los puentes de hidrógeno entre bases y abre la doble hélice formando la horquilla de replicación (con ayuda de topoisomerasas, que alivian la tensión, y proteínas SSB, que estabilizan las hebras sueltas). |
| **Primasa** | Sintetiza un cebador corto de **ARN** que aporta el extremo 3'-OH libre, ya que la ADN polimerasa no puede empezar de cero. |
| **ADN polimerasa** | Añade desoxirribonucleótidos al extremo 3' (síntesis siempre 5'→3'). Hebra **continua** (líder) en un solo tramo; hebra **discontinua** (retrasada) en fragmentos de Okazaki. Tiene actividad correctora exonucleasa 3'→5'. En procariotas, la Pol I retira los cebadores de ARN y los sustituye por ADN. |
| **Ligasa** | Une los fragmentos de Okazaki formando el enlace fosfodiéster que falta, dejando una hebra continua. |

### Reflexión: error no corregido de la ADN polimerasa
Si se incorpora una base incorrecta y fallan tanto la corrección de pruebas (exonucleasa 3'→5') como la reparación de apareamientos erróneos (*mismatch repair*), en la siguiente ronda de replicación la base errónea sirve de molde y el cambio queda fijado en **ambas hebras**: es una **mutación puntual permanente y heredable** por las células hijas. Sus efectos dependen de dónde caiga: silenciosa (mismo aminoácido), de sentido erróneo (otro aminoácido), sin sentido (codón de paro prematuro) o sin efecto si está en una región no funcional. Si afecta a genes de control del ciclo celular, puede contribuir a enfermedades como el cáncer.

In [ ]:
# --- Extensión con Biopython: hebra complementaria automática ---
superior = Seq("ATGCCGTTAGCT")        # 5'->3'
inferior_manual = "TACGGCAATCGA"      # tal como aparece en el enunciado (3'->5')

complementaria = superior.complement()             # alineada 3'->5'
complementaria_5a3 = superior.reverse_complement() # misma hebra escrita 5'->3'

print("Hebra molde      :", codones(superior))
print("Complementaria   :", codones(complementaria), "(3'->5')")
print("Complementaria   :", codones(complementaria_5a3), "(5'->3')")
print("¿Coincide con el resultado manual?", str(complementaria) == inferior_manual)

In [ ]:
def replicar(superior):
    # Una ronda de replicación semiconservativa
    inferior = superior.complement()          # hebra parental inferior (3'->5')
    nueva_1 = superior.complement()           # se sintetiza sobre la superior
    nueva_2 = inferior.complement()           # se sintetiza sobre la inferior
    print("Molécula parental:"); duplex(superior, inferior)
    print("\nHelicasa separa las hebras -> cada una hace de molde\n")
    print("Molécula hija 1:"); duplex(superior, nueva_1, "(parental)", "(NUEVA)")
    print("\nMolécula hija 2:"); duplex(nueva_2, inferior, "(NUEVA)", "(parental)")
    iguales = (str(nueva_2) == str(superior)) and (str(nueva_1) == str(inferior))
    print("\n¿Hijas idénticas a la parental?", iguales)
    return (superior, nueva_1), (nueva_2, inferior)

hija1, hija2 = replicar(superior)

In [ ]:
def simular_horquilla(superior, fragmento=4, cebador=2):
    # Simulación didáctica de la horquilla (avanza de izquierda a derecha)
    sup = str(superior)
    print("1) HELICASA: abre la doble hélice por el extremo izquierdo.\n")

    # Hebra líder: se sintetiza sobre la inferior (3'->5'), crece en el sentido de la horquilla
    lider = sup  # la nueva hebra líder tiene la secuencia de la superior (5'->3')
    print("2) HEBRA LÍDER (molde = hebra inferior 3'->5'):")
    print(f"   PRIMASA  -> cebador ARN 5'-{Seq(lider[:cebador]).transcribe()}-3'")
    print(f"   ADN POL  -> síntesis continua 5'-{lider[cebador:]}-3' desde el cebador")
    print(f"   Resultado: 5'-{lider}-3' (tras sustituir el cebador por ADN)\n")

    # Hebra retrasada: molde = superior; crece en sentido contrario a la horquilla
    print("3) HEBRA RETRASADA (molde = hebra superior 5'->3'): fragmentos de Okazaki")
    frags = []
    for i in range(0, len(sup), fragmento):
        seg = Seq(sup[i:i+fragmento])
        frag = seg.reverse_complement()               # 5'->3'
        frags.append(frag)
        print(f"   Horquilla abre posiciones {i+1}-{i+len(seg)}: "
              f"PRIMASA cebador 5'-{frag[:cebador].transcribe()}-3' + ADN POL -> "
              f"fragmento 5'-{frag}-3'")
    print("   ADN POL I (procariotas) / RNasa H + Pol δ (eucariotas): elimina los cebadores de ARN y rellena con ADN")
    retrasada = Seq("".join(str(f) for f in reversed(frags)))
    print(f"4) LIGASA: une {len(frags)} fragmentos -> 5'-{retrasada}-3'")
    ok = str(retrasada) == str(superior.reverse_complement())
    print("\n¿La hebra retrasada ligada es la complementaria correcta?", ok)

simular_horquilla(superior)

In [ ]:
# Reflexión con código: error NO corregido de la polimerasa y su fijación en la siguiente ronda
def replicar_con_error(molde, pos, base_erronea):
    nueva = list(str(molde.complement()))
    print(f"Base correcta en pos {pos+1}: {nueva[pos]}  -> la polimerasa pone {base_erronea}")
    nueva[pos] = base_erronea
    return Seq("".join(nueva))

nueva_con_error = replicar_con_error(superior, pos=4, base_erronea="T")   # debería ser G (frente a C) y pone T
print("Ronda 1 (desapareamiento):")
duplex(superior, nueva_con_error, "(parental)", "(nueva con error)")
print("\nRonda 2: la hebra con error hace de molde -> mutación fijada en ambas hebras:")
mutante_sup = nueva_con_error.complement()
duplex(mutante_sup, nueva_con_error)
print("\nProteína original:", superior.translate(), "| mutante:", mutante_sup.translate())

---
## Ejercicio 2. Transcripción del ADN a ARN

```
5' – ATG CCT GAA TGC – 3'
3' – TAC GGA CTT ACG – 5'
```

### Resolución manual
- **Cadena molde:** la inferior, `3'-TAC GGA CTT ACG-5'`. La ARN polimerasa la lee 3'→5' y sintetiza el ARN 5'→3'. La superior es la **cadena codificante** (tiene la misma secuencia que el ARNm, con T en lugar de U) y empieza por el codón de inicio ATG.
- **Transcrito:** `5' – AUG CCU GAA UGC – 3'` (complementario a la molde, U en lugar de T).
- *(Traducido sería Met–Pro–Glu–Cys; al no haber codón de paro, el fragmento es parte de una región codificante mayor.)*

**Región promotora y región codificante:**
- La **región promotora** está *aguas arriba* (hacia el 5' de la cadena codificante) del inicio de la transcripción y **no aparece** en el fragmento dado. Es donde se unen la ARN polimerasa y los factores de transcripción; en eucariotas suele contener la **caja TATA** (~ −25/−30 pb), y en procariotas las cajas −10 (TATAAT, caja de Pribnow) y −35 (TTGACA). El promotor no se traduce.
- La **región codificante** empieza en el codón **ATG** (AUG en el ARNm) y se extiende, en el mismo marco de lectura, hasta un codón de paro. En este fragmento, las 12 bases corresponden a región codificante.

In [ ]:
# Crear el FASTA de ejemplo (cadena codificante 5'->3')
rec = SeqRecord(Seq("ATGCCTGAATGC"), id="ej2", description="cadena codificante 5'->3'")
SeqIO.write(rec, "ej2.fasta", "fasta")
print(open("ej2.fasta").read())

In [ ]:
def fasta_a_arnm(ruta, hebra="codificante"):
    # Lee un FASTA (5'->3') y devuelve el ARNm 5'->3'.
    # hebra='codificante': el ARNm es la misma secuencia con U.
    # hebra='molde': el ARNm es el complementario inverso de la molde.
    registro = next(SeqIO.parse(ruta, "fasta"))
    adn = registro.seq.upper()
    if hebra == "codificante":
        arnm = adn.transcribe()
    elif hebra == "molde":
        arnm = adn.reverse_complement().transcribe()
    else:
        raise ValueError("hebra debe ser 'codificante' o 'molde'")
    print(f"[{registro.id}] ADN ({hebra}) 5'-{codones(adn)}-3'  ->  ARNm 5'-{codones(arnm)}-3'")
    return arnm

arnm = fasta_a_arnm("ej2.fasta", "codificante")
print("¿Coincide con el resultado manual?", str(arnm) == "AUGCCUGAAUGC")
print("Traducción:", seq3(str(arnm.translate()), custom_map={"*": "Stop"}))

In [ ]:
# Experimento: cambiar la orientación de la hebra
molde_5a3 = Seq("ATGCCTGAATGC").reverse_complement()   # molde escrita 5'->3' = GCATTCAGGCAT
molde_3a5 = Seq("TACGGACTTACG")                         # molde pegada tal cual (3'->5') como si fuera 5'->3'

casos = {
    "codificante 5'->3' (correcto)": (Seq("ATGCCTGAATGC"), "codificante"),
    "molde 5'->3' (correcto)":        (molde_5a3, "molde"),
    "molde tratada como codificante": (molde_5a3, "codificante"),
    "molde 3'->5' pegada al revés":   (molde_3a5, "molde"),
}
filas = []
for nombre, (s, hebra) in casos.items():
    SeqIO.write(SeqRecord(s, id="tmp", description=""), "tmp.fasta", "fasta")
    a = fasta_a_arnm("tmp.fasta", hebra)
    filas.append({"caso": nombre, "ARNm": codones(a), "proteína": str(a.translate()),
                  "correcto": str(a) == "AUGCCUGAAUGC"})
pd.DataFrame(filas)

**Observación:** solo se obtiene el ARNm correcto (`AUG CCU GAA UGC`, Met-Pro-Glu-Cys) cuando se indica bien qué hebra es y se respeta la orientación 5'→3'. Si se toma la molde como codificante o se pega una hebra en orientación 3'→5', se obtiene otra secuencia completamente distinta, sin codón AUG inicial y con una proteína sin sentido biológico. Por eso los FASTA siempre se escriben 5'→3' y es fundamental saber de qué hebra se trata.

---
## Ejercicio 3. Traducción del ARNm a proteína

Transcrito: `5' – AUG UAU GCU UAA – 3'`

### Resolución manual
| Codón | AUG | UAU | GCU | UAA |
|---|---|---|---|---|
| Aminoácido | **Met (M)** – inicio | Tyr (Y) | Ala (A) | **Stop** – paro (ocre) |

- **Codón de inicio:** AUG (Met). **Codón de paro:** UAA.
- **Péptido:** `NH₂ – Met – Tyr – Ala – COOH` (el codón de paro no codifica aminoácido; lo reconoce un factor de liberación).

### Reflexión
- **AUG → GUG:** en eucariotas el ribosoma, que recorre el ARNm desde el 5' buscando un AUG, no reconoce eficientemente GUG como inicio (en ese contexto codifica Val). Seguiría buscando el siguiente AUG: aquí aparece uno en `GUG U`**`AUG`**`CU UAA`, pero en **otro marco de lectura** (+1), dando Met–Leu… sin codón de paro en fase → proteína distinta/aberrante o ausencia de proteína funcional. En procariotas GUG sí puede funcionar como inicio alternativo (menos eficiente) y se incorporaría igualmente fMet.
- **Pérdida del codón de paro (mutación *non-stop*, p. ej. UAA → CAA):** el ribosoma no se detiene y sigue traduciendo la región 3' UTR hasta encontrar otro codón de paro en fase, produciendo una proteína **más larga** con aminoácidos extra en el extremo C, que puede plegarse mal o perder función. Si no encuentra ningún paro, llega a la cola poli-A (traduce lisinas, AAA) y el mensajero suele degradarse por el mecanismo *non-stop decay*.

In [ ]:
arnm3 = Seq("AUGUAUGCUUAA")
print("Codones   :", codones(arnm3))
print("Inicio    :", arnm3[:3], "->", seq3(str(arnm3[:3].translate())))
print("Paro      :", arnm3[-3:], "->", "Stop" if arnm3[-3:].translate() == "*" else "no")
proteina = arnm3.translate(to_stop=True)
print("Proteína  :", proteina, "=", seq3(str(proteina), custom_map={"*": "Stop"}))
print("cds=True  :", arnm3.translate(cds=True), "(Biopython valida que empieza por inicio y termina en paro)")
print("¿Coincide con el resultado manual?", str(proteina) == "MYA")

In [ ]:
tabla = CodonTable.unambiguous_rna_by_id[1]
print("Tabla estándar -> codones de inicio:", tabla.start_codons, "| de paro:", tabla.stop_codons)
tabla11 = CodonTable.unambiguous_rna_by_id[11]
print("Tabla bacteriana (11) -> codones de inicio:", tabla11.start_codons)

In [ ]:
# Mutación 1: AUG -> GUG
mut_inicio = Seq("GUGUAUGCUUAA")
print("Leído desde el codón 1 (tabla estándar):", mut_inicio.translate(), "-> GUG se lee como Val")
try:
    mut_inicio.translate(cds=True)
except Exception as e:
    print("Como CDS eucariota:", e)
print("Como CDS bacteriana (tabla 11):", mut_inicio.translate(table=11, cds=True), "-> GUG inicia con Met")

# Modelo de escaneo eucariota: el ribosoma busca el primer AUG
pos = mut_inicio.find("AUG")
print(f"\nPrimer AUG en posición {pos+1} (marco {pos % 3 + 1} en vez de 1)")
orf = mut_inicio[pos:]
orf = orf[: len(orf) - len(orf) % 3]
print("Traducción desde ese AUG:", orf.translate(), "-> otro marco, sin paro en fase: proteína aberrante")

In [ ]:
# Mutación 2: pérdida del codón de paro (UAA -> CAA), con una 3' UTR hipotética
utr3 = "GCAGGAUUCUGAACCAAAAAAAAAAAA"          # 3' UTR inventada + cola poli-A
normal  = Seq("AUGUAUGCUUAA" + utr3)
nonstop = Seq("AUGUAUGCUCAA" + utr3)
print("Normal  :", normal.translate(to_stop=True))
print("Non-stop:", nonstop.translate(to_stop=True), "-> la traducción continúa hasta el siguiente paro (UGA)")

sin_paro = Seq("AUGUAUGCUCAA" + "GCAGGAUUC" + "A"*18)   # sin paro en fase: llega a la poli-A
print("Sin paro en fase:", sin_paro.translate(), "-> lisinas de la poli-A (non-stop decay)")

---
## Ejercicio 4. Splicing alternativo

Gen: `Exón 1 – Exón 2 – Exón 3 – Exón 4 – Exón 5` (con intrones entre ellos).

### Combinaciones propuestas
| Isoforma | Exones | Tipo de evento |
|---|---|---|
| A (completa) | 1-2-3-4-5 | Todos los exones |
| B | 1-2-4-5 | *Exon skipping* del exón 3 |
| C | 1-3-5 | Omisión de los exones 2 y 4 |
| D | 1-2-5 | Omisión de los exones 3 y 4 |

### Diferencias esperadas en las proteínas
- **Longitud y dominios:** si el exón omitido codifica un dominio (unión a ligando, transmembrana, sitio catalítico, señal de localización…), la isoforma lo pierde y puede cambiar su afinidad, localización o actividad.
- **Marco de lectura:** si la longitud del exón omitido es **múltiplo de 3**, se conserva el marco y la proteína solo pierde esos aminoácidos. Si **no** lo es, se produce un **desplazamiento del marco**: todo lo que va detrás cambia y suele aparecer un codón de paro prematuro (proteína truncada, o ARNm degradado por *nonsense-mediated decay*).
- **Regulación:** isoformas distintas pueden expresarse en tejidos o etapas diferentes, o incluso tener funciones antagonistas (p. ej. versiones solubles que actúan como "señuelo" de un receptor).

### Reflexión
Con un solo gen se pueden producir muchas proteínas, porque las combinaciones de exones crecen rápidamente (con *n* exones alternativos independientes hay hasta 2ⁿ variantes). Así, los ~20 000 genes humanos codifican un proteoma mucho mayor (más del 90 % de los genes multiexónicos presentan splicing alternativo). Es una forma económica de diversificar funciones y regularlas por tejido, sin necesidad de duplicar genes.

In [ ]:
# Gen de juguete: exones con longitudes elegidas para ver el efecto sobre el marco de lectura
exones = {
    1: "ATGGCTGAACGT",      # 12 nt (incluye ATG)
    2: "AAACTGGGTCCT",      # 12 nt
    3: "GATCCGCAAGT",       # 11 nt  (no múltiplo de 3)
    4: "TCGATGCCTATGG",     # 13 nt  (no múltiplo de 3; 3+4 = 24 nt)
    5: "CGAAACCCGTGGTAAACTGATAG",  # incluye codón de paro
}
for n, s in exones.items():
    nota = "(incluye codón de paro y 3' UTR)" if n == 5 else ("(múltiplo de 3)" if len(s) % 3 == 0 else "(NO múltiplo de 3)")
    print(f"Exón {n}: {len(s):2d} nt  {nota}")

isoformas = {"A": [1,2,3,4,5], "B": [1,2,4,5], "C": [1,3,5], "D": [1,2,5]}
filas = []
for nombre, combo in isoformas.items():
    arn = Seq("".join(exones[e] for e in combo)).transcribe()
    arn_marco = arn[: len(arn) - len(arn) % 3]
    omitido = sum(len(exones[e]) for e in exones if e not in combo)
    prot = arn_marco.translate(to_stop=True)
    filas.append({"isoforma": nombre, "exones": "-".join(map(str, combo)), "nt": len(arn),
                  "nt omitidos": omitido, "marco conservado": omitido % 3 == 0,
                  "proteína": str(prot), "aa": len(prot)})
pd.DataFrame(filas)

Las isoformas **A** y **D** mantienen el marco (en D se omiten 24 nt = 8 aa). **B** y **C** omiten un número de nucleótidos que no es múltiplo de 3: el marco se desplaza y el C-terminal cambia por completo.

### Extensión: FGFR2 en Ensembl
FGFR2 (receptor 2 del factor de crecimiento de fibroblastos) es un ejemplo clásico: el splicing **mutuamente excluyente** de dos exones del tercer dominio tipo inmunoglobulina genera las isoformas **FGFR2-IIIb** (epitelial, une FGF7/FGF10) y **FGFR2-IIIc** (mesenquimal, une FGF2/FGF18). Un cambio de unos 50 aminoácidos en el sitio de unión al ligando cambia qué FGF reconoce el receptor; el cambio anómalo entre isoformas se ha relacionado con transición epitelio-mesénquima y cáncer. Otras isoformas carecen del dominio transmembrana (formas solubles) o de dominios Ig.

La celda siguiente consulta la **API REST de Ensembl** y compara los transcritos codificantes.

In [ ]:
def ensembl_transcritos(simbolo="FGFR2", especie="homo_sapiens"):
    url = f"https://rest.ensembl.org/lookup/symbol/{especie}/{simbolo}?expand=1;content-type=application/json"
    with urllib.request.urlopen(url, timeout=20) as r:
        gen = json.load(r)
    filas = []
    for t in gen["Transcript"]:
        tr = t.get("Translation")
        filas.append({"transcrito": t["id"], "nombre": t.get("display_name"), "biotipo": t["biotype"],
                      "canónico": bool(t.get("is_canonical")), "exones": len(t.get("Exon", [])),
                      "aa": tr["length"] if tr else None})
    df = pd.DataFrame(filas)
    print(f"{simbolo} ({gen['id']}), cromosoma {gen['seq_region_name']}: {len(df)} transcritos")
    return df

try:
    df_fgfr2 = ensembl_transcritos("FGFR2")
    codif = df_fgfr2[df_fgfr2.biotipo == "protein_coding"].sort_values("aa", ascending=False)
    display(codif.head(15))
    print("Rango de longitudes de proteína:", codif.aa.min(), "-", codif.aa.max(), "aa")
except Exception as e:
    print("No se pudo acceder a Ensembl (¿sin conexión?):", e)
    print("Consulta manual: https://www.ensembl.org/Homo_sapiens/Gene/Summary?g=ENSG00000066468")

---
## Ejercicio 5. Introducción a las proteínas

Secuencia: `Met – Ile – Ser – Gly – Val – Lys – His`

### Resolución manual
- **Extremo N (amino-terminal):** **Met**, primer aminoácido, con el grupo –NH₃⁺ libre (es el que codifica el AUG de inicio).
- **Extremo C (carboxilo-terminal):** **His**, último aminoácido, con el grupo –COO⁻ libre.
- Por convención, las secuencias se escriben y se sintetizan de N a C.

### Reflexión
- **El orden de los aminoácidos (estructura primaria) determina el plegamiento** (principio de Anfinsen): las propiedades de las cadenas laterales (tamaño, carga, polaridad, capacidad de formar puentes de hidrógeno o disulfuro) definen qué segmentos forman **hélices α** o **láminas β** (estructura secundaria) y cómo se empaqueta todo en 3D (terciaria). En medio acuoso, los residuos **hidrofóbicos** (Met, Ile, Val…) tienden a quedar en el **núcleo** y los **hidrofílicos** (Ser, Lys, His…) en la **superficie**. Gly, por su flexibilidad, suele aparecer en giros.
- **Hidrofóbico → hidrofílico en el interior** (p. ej. Val → Lys): se introduce una carga o un grupo polar en un entorno apolar, lo que desestabiliza el núcleo hidrofóbico. La proteína puede plegarse mal, perder estabilidad o actividad, agregarse o degradarse. El caso contrario en la superficie también es grave: en la **anemia falciforme** (hemoglobina, Glu6Val) un residuo hidrofílico superficial pasa a hidrofóbico y provoca la polimerización de la hemoglobina.

In [ ]:
pep = "MISGVKH"
print("Secuencia (1 letra):", pep)
print("Secuencia (3 letras):", "-".join(seq3(a) for a in pep))
print(f"Extremo N: {seq3(pep[0])}  |  Extremo C: {seq3(pep[-1])}\n")

tabla_kd = pd.DataFrame({"posición": range(1, len(pep)+1), "aa": [seq3(a) for a in pep],
                         "hidropatía KD": [kd[a] for a in pep]})
tabla_kd["carácter"] = tabla_kd["hidropatía KD"].apply(lambda v: "hidrofóbico" if v > 0 else "hidrofílico")
display(tabla_kd)

def resumen(p):
    pa = ProteinAnalysis(p)
    return {"secuencia": p, "GRAVY": round(pa.gravy(), 3), "pI": round(pa.isoelectric_point(), 2),
            "masa (Da)": round(pa.molecular_weight(), 1)}

# Mutación hidrofóbico -> hidrofílico en una posición "interna" (Val5 -> Lys / Asp)
mutantes = [pep, pep[:4] + "K" + pep[5:], pep[:4] + "D" + pep[5:]]
pd.DataFrame([resumen(p) for p in mutantes], index=["original", "V5K", "V5D"])

Un GRAVY menor indica una secuencia más hidrofílica: al cambiar Val por Lys o Asp se pierde hidrofobicidad y cambian la carga y el punto isoeléctrico, lo que en el núcleo de una proteína real desestabilizaría el plegamiento.

### Extensión: estructura en el PDB
Se usa la **ubiquitina humana (PDB 1UBQ)**, pequeña (76 aa) y con hélice α y lámina β bien definidas. Se leen los registros `HELIX` y `SHEET` del fichero PDB para asignar la estructura secundaria a cada residuo.

In [ ]:
from Bio.PDB import PDBParser, PPBuilder
import io

def estructura_secundaria(pdb_id="1UBQ"):
    url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    texto = urllib.request.urlopen(url, timeout=20).read().decode()
    estructura = PDBParser(QUIET=True).get_structure(pdb_id, io.StringIO(texto))
    cadena = next(estructura[0].get_chains())
    residuos = [r for r in cadena if r.id[0] == " "]
    seq = "".join(str(pp.get_sequence()) for pp in PPBuilder().build_peptides(cadena))
    ss = {r.id[1]: "-" for r in residuos}
    for linea in texto.splitlines():
        if linea.startswith("HELIX"):
            ini, fin, tipo = int(linea[21:25]), int(linea[33:37]), "H"
        elif linea.startswith("SHEET"):
            ini, fin, tipo = int(linea[22:26]), int(linea[33:37]), "E"
        else:
            continue
        for i in range(ini, fin + 1):
            if i in ss: ss[i] = tipo
    ss_str = "".join(ss[r.id[1]] for r in residuos)
    titulo = next((l[10:].strip() for l in texto.splitlines() if l.startswith("TITLE")), "")
    print(f"{pdb_id}: {titulo}  |  cadena {cadena.id}, {len(residuos)} residuos\n")
    for i in range(0, len(seq), 60):
        print("Secuencia:", seq[i:i+60]); print("Estructura:", ss_str[i:i+60]); print()
    print(f"H = hélice α ({ss_str.count('H')} res.), E = lámina β ({ss_str.count('E')} res.), - = giro/bucle")
    return seq, ss_str

try:
    seq_ubq, ss_ubq = estructura_secundaria("1UBQ")
    comp = pd.DataFrame({"aa": list(seq_ubq), "ss": list(ss_ubq)})
    comp["hidropatía"] = comp.aa.map(kd)
    print("\nHidropatía media por tipo de estructura:")
    print(comp.groupby("ss")["hidropatía"].mean().round(2).rename({"H": "hélice", "E": "lámina", "-": "bucle"}))
except Exception as e:
    print("No se pudo descargar del PDB (¿sin conexión?):", e)
    print("Consulta manual: https://www.rcsb.org/structure/1UBQ")

**Reflexión sobre mutaciones puntuales y plegamiento:** una sola sustitución puede romper una hélice (p. ej. introducir **Pro**, que no puede formar el puente de hidrógeno del esqueleto y dobla la cadena), impedir el apilamiento de las hebras de una lámina β, eliminar un puente disulfuro o insertar una carga en el núcleo hidrofóbico. El resultado puede ser una proteína inestable, mal plegada o propensa a agregarse (como en algunas enfermedades neurodegenerativas), aunque el resto de la secuencia sea idéntico. En cambio, sustituciones conservativas en la superficie (p. ej. Lys → Arg) suelen tolerarse.

---
## Ejercicio 6. Actividad integradora: del ADN a la proteína

1. **Secuencia:** se descarga de **NCBI Nucleotide** la región codificante (CDS) de la **insulina humana (gen *INS*, RefSeq `NM_000207`)** en formato FASTA. Si no hay conexión, se usa una secuencia de respaldo.
2. **Pipeline** (`dogma_pipeline.py`, se genera desde este notebook para subirlo al repositorio): replicación → transcripción → búsqueda del ORF → traducción, **informando en todo momento** de lo que ocurre mediante `logging`.
3. **Reflexión** apoyada en un experimento de mutaciones aleatorias en cada etapa.

In [ ]:
%%writefile dogma_pipeline.py
"""
dogma_pipeline.py — Pipeline del dogma central con Biopython.
Replicación (hebras complementarias) -> Transcripción (ARNm) -> Traducción (proteína).

Uso:
    python dogma_pipeline.py secuencia.fasta [--hebra codificante|molde] [--tabla 1] [--salida prefijo]
"""
import argparse
import logging
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.SeqUtils.ProtParam import ProteinAnalysis

log = logging.getLogger("dogma")


def configurar_log(nivel=logging.INFO):
    if not log.handlers:
        h = logging.StreamHandler()
        h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
        log.addHandler(h)
    log.setLevel(nivel)
    log.propagate = False


def corta(s, n=45):
    s = str(s)
    return s if len(s) <= n else f"{s[:n]}... ({len(s)} nt/aa)"


class PipelineDogmaCentral:
    def __init__(self, secuencia, nombre="secuencia", hebra="codificante", tabla=1):
        self.nombre, self.hebra, self.tabla = nombre, hebra, tabla
        self.adn = Seq(str(secuencia).upper().replace("U", "T"))
        self.resultados = {}
        invalidas = set(str(self.adn)) - set("ACGTN")
        if invalidas:
            raise ValueError(f"Caracteres no válidos en la secuencia: {invalidas}")
        log.info("=" * 70)
        log.info(f"Secuencia '{nombre}' cargada: {len(self.adn)} nt, hebra '{hebra}'")
        gc = 100 * (self.adn.count("G") + self.adn.count("C")) / max(len(self.adn), 1)
        log.info(f"Contenido GC: {gc:.1f} %")

    @classmethod
    def desde_fasta(cls, ruta, **kw):
        reg = next(SeqIO.parse(ruta, "fasta"))
        log.info(f"Leyendo FASTA '{ruta}': {reg.description}")
        return cls(reg.seq, nombre=reg.id, **kw)

    # ---------- PASO 1: REPLICACIÓN ----------
    def replicacion(self):
        log.info("-" * 70)
        log.info("PASO 1 · REPLICACIÓN (semiconservativa)")
        sup = self.adn if self.hebra == "codificante" else self.adn.reverse_complement()
        inf = sup.reverse_complement()  # hebra complementaria escrita 5'->3'
        log.info("Helicasa: separa la doble hélice en dos hebras molde")
        log.info("Primasa: coloca cebadores de ARN; ADN polimerasa: sintetiza 5'->3'")
        nueva_1 = sup.reverse_complement()   # sintetizada sobre la hebra superior
        nueva_2 = inf.reverse_complement()   # sintetizada sobre la hebra inferior
        log.info(f"Hebra parental 5'->3'           : {corta(sup)}")
        log.info(f"Hebra parental complementaria   : {corta(inf)}")
        log.info(f"Nueva hebra (molde = superior)  : {corta(nueva_1)}")
        log.info(f"Nueva hebra (molde = inferior)  : {corta(nueva_2)}")
        log.info("Ligasa: une los fragmentos de Okazaki de la hebra retrasada")
        ok = str(nueva_1) == str(inf) and str(nueva_2) == str(sup)
        log.info(f"Comprobación: moléculas hijas idénticas a la parental -> {ok}")
        self.codificante, self.molde = sup, inf
        self.resultados["replicacion"] = {"hija_1": (sup, nueva_1), "hija_2": (nueva_2, inf), "ok": ok}
        return self.resultados["replicacion"]

    # ---------- PASO 2: TRANSCRIPCIÓN ----------
    def transcripcion(self):
        if "replicacion" not in self.resultados:
            self.replicacion()
        log.info("-" * 70)
        log.info("PASO 2 · TRANSCRIPCIÓN")
        log.info("ARN polimerasa lee la cadena MOLDE 3'->5' y sintetiza ARN 5'->3' (U en lugar de T)")
        arnm = self.molde.reverse_complement().transcribe()
        log.info(f"Cadena molde 5'->3' : {corta(self.molde)}")
        log.info(f"ARNm 5'->3'         : {corta(arnm)}")
        log.info(f"Comprobación: ARNm == codificante con U -> {str(arnm) == str(self.codificante.transcribe())}")
        self.arnm = arnm
        self.resultados["transcripcion"] = arnm
        return arnm

    # ---------- búsqueda de ORF ----------
    def buscar_orf(self):
        tabla_stop = {"TAA", "TAG", "TGA", "UAA", "UAG", "UGA"}
        mejor = None
        s = str(self.arnm)
        for marco in range(3):
            i = marco
            while i + 3 <= len(s):
                if s[i:i+3] == "AUG":
                    j = i
                    while j + 3 <= len(s) and s[j:j+3] not in tabla_stop:
                        j += 3
                    fin = j + 3 if j + 3 <= len(s) else j
                    if mejor is None or fin - i > mejor[1] - mejor[0]:
                        mejor = (i, fin, marco, j + 3 <= len(s))
                    i = j
                i += 3
        if mejor is None:
            log.warning("No se ha encontrado ningún codón AUG: no hay ORF")
            return None
        ini, fin, marco, con_stop = mejor
        log.info(f"ORF más largo: posiciones {ini+1}-{fin}, marco {marco+1}, "
                 f"{'con' if con_stop else 'SIN'} codón de paro")
        return self.arnm[ini:fin]

    # ---------- PASO 3: TRADUCCIÓN ----------
    def traduccion(self):
        if "transcripcion" not in self.resultados:
            self.transcripcion()
        log.info("-" * 70)
        log.info("PASO 3 · TRADUCCIÓN")
        orf = self.buscar_orf()
        if orf is None:
            self.proteina = Seq("")
            return self.proteina
        log.info(f"Codón de inicio: {orf[:3]} (Met) -> el ribosoma ensambla y empieza en el sitio P")
        log.info(f"Codón de paro  : {orf[-3:]} -> factor de liberación termina la traducción")
        orf3 = orf[: len(orf) - len(orf) % 3]
        prot = orf3.translate(table=self.tabla, to_stop=True)
        log.info(f"Codones leídos : {len(orf3)//3}  ->  {len(prot)} aminoácidos")
        log.info(f"Proteína (N->C): {corta(prot, 60)}")
        if prot:
            pa = ProteinAnalysis(str(prot).replace("X", ""))
            log.info(f"Masa ≈ {pa.molecular_weight()/1000:.2f} kDa | pI ≈ {pa.isoelectric_point():.2f} | "
                     f"GRAVY = {pa.gravy():.3f}")
        self.proteina = prot
        self.resultados["traduccion"] = prot
        return prot

    def ejecutar(self):
        self.replicacion()
        self.transcripcion()
        self.traduccion()
        log.info("=" * 70)
        log.info("Pipeline completado")
        return self.resultados

    def guardar(self, prefijo):
        recs = [
            SeqRecord(self.codificante, id=f"{self.nombre}_codificante", description="ADN 5'->3'"),
            SeqRecord(self.molde, id=f"{self.nombre}_molde", description="ADN complementario 5'->3'"),
            SeqRecord(self.arnm, id=f"{self.nombre}_ARNm", description="ARNm 5'->3'"),
            SeqRecord(self.proteina, id=f"{self.nombre}_proteina", description="N->C"),
        ]
        ruta = f"{prefijo}_resultados.fasta"
        SeqIO.write(recs, ruta, "fasta")
        log.info(f"Resultados guardados en {ruta}")
        return ruta


def main():
    ap = argparse.ArgumentParser(description="Replicación -> Transcripción -> Traducción")
    ap.add_argument("fasta")
    ap.add_argument("--hebra", default="codificante", choices=["codificante", "molde"])
    ap.add_argument("--tabla", type=int, default=1, help="tabla de código genético NCBI")
    ap.add_argument("--salida", default=None, help="prefijo para guardar resultados FASTA")
    a = ap.parse_args()
    configurar_log()
    p = PipelineDogmaCentral.desde_fasta(a.fasta, hebra=a.hebra, tabla=a.tabla)
    p.ejecutar()
    if a.salida:
        p.guardar(a.salida)


if __name__ == "__main__":
    main()

In [ ]:
# Descargar la CDS de la insulina humana desde NCBI (poner tu correo: lo exige NCBI)
Entrez.email = "tu_correo@ejemplo.com"
ruta_fasta = "INS_cds.fasta"
try:
    with Entrez.efetch(db="nucleotide", id="NM_000207", rettype="fasta_cds_na", retmode="text") as h:
        texto = h.read()
    open(ruta_fasta, "w").write(texto)
    print("Descargado de NCBI:", texto.splitlines()[0][:120])
except Exception as e:
    print("Sin acceso a NCBI (", e, ") -> se usa una secuencia de respaldo")
    respaldo = "ATGCCGTTAGCTCCTGAATGCTATGCTGGCAAACTGTAA"   # ORF construido con los ejercicios 1-3
    SeqIO.write(SeqRecord(Seq(respaldo), id="respaldo", description="ORF de ejemplo"), ruta_fasta, "fasta")
print(open(ruta_fasta).read()[:400])

In [ ]:
import importlib, dogma_pipeline
importlib.reload(dogma_pipeline)
dogma_pipeline.configurar_log()

pipe = dogma_pipeline.PipelineDogmaCentral.desde_fasta(ruta_fasta)
res = pipe.ejecutar()
pipe.guardar("INS")

In [ ]:
# Comprobación: con la insulina, la preproinsulina humana empieza por MALWMRLLPLLALLALWGPDPAAA (péptido señal)
print("Proteína obtenida:", pipe.proteina)
if len(pipe.proteina) > 100:
    print("¿Empieza como la preproinsulina (MALWMRLL...)?", str(pipe.proteina).startswith("MALWMRLL"))

In [ ]:
# El mismo pipeline también se puede ejecutar desde la terminal (así se usará desde el repositorio)
!python dogma_pipeline.py ej2.fasta --salida ej2

### Reflexión: ¿qué punto del proceso es más vulnerable?
Para responder con datos, se introducen **mutaciones puntuales aleatorias** en la CDS y se clasifica el efecto sobre la proteína (silenciosa, de sentido erróneo, sin sentido o pérdida del paro). Después se compara **dónde** ocurre el error.

In [ ]:
def efecto_mutacion(cds, n=2000, semilla=1):
    rng = random.Random(semilla)
    s = str(cds)
    wt = Seq(s).translate(to_stop=True)
    conteo = {"silenciosa": 0, "sentido erróneo": 0, "sin sentido (paro prematuro)": 0,
              "pérdida de inicio": 0, "pérdida de paro": 0}
    for _ in range(n):
        i = rng.randrange(len(s))
        base = rng.choice([b for b in "ACGT" if b != s[i]])
        mut = s[:i] + base + s[i+1:]
        if i < 3:
            conteo["pérdida de inicio"] += 1; continue
        p = Seq(mut).translate(to_stop=True)
        if p == wt:
            conteo["silenciosa"] += 1
        elif len(p) < len(wt):
            conteo["sin sentido (paro prematuro)"] += 1
        elif len(p) > len(wt):
            conteo["pérdida de paro"] += 1
        else:
            conteo["sentido erróneo"] += 1
    return pd.Series(conteo).div(n).mul(100).round(1).rename("% de mutaciones")

cds = pipe.codificante
efecto_mutacion(cds).to_frame()

**Conclusión.** El tipo de efecto (≈ 20-25 % silenciosas por la degeneración del código, la mayoría de sentido erróneo y una fracción sin sentido) es el mismo esté donde esté el error, porque depende del código genético. Lo que cambia es **su persistencia**:

| Etapa | ¿Dónde queda el error? | Alcance |
|---|---|---|
| **Replicación** | En el ADN, se copia en cada división | **Permanente y heredable**: todas las moléculas de ARNm y todas las proteínas de esa célula y sus descendientes (y de la descendencia si ocurre en la línea germinal) |
| Transcripción | En una molécula de ARNm | Transitorio: solo las proteínas producidas a partir de ese ARNm; el resto de transcritos son correctos |
| Traducción | En una sola cadena polipeptídica | Mínimo: una única molécula de proteína, que suele degradarse |

Por tanto, el punto **más vulnerable** en cuanto a consecuencias para la función de la proteína es la **replicación del ADN** (y, en general, cualquier daño en el ADN no reparado), aunque la ADN polimerasa sea la enzima más fiel (~1 error por 10⁹-10¹⁰ bases gracias a la corrección y la reparación), frente a ~10⁻⁵ de la ARN polimerasa y ~10⁻⁴ del ribosoma. Dentro de la secuencia, las posiciones más críticas son el **codón de inicio**, los codones que pueden convertirse en **paro prematuro**, las **inserciones/deleciones** que desplazan el marco y las **señales de splicing**, además de los residuos del núcleo hidrofóbico o del sitio activo.

In [ ]:
# Demostración de la persistencia: el mismo error introducido en el ADN o en un ARNm
cds_str = str(pipe.codificante)
pos = next(i for i in range(3, len(cds_str) - 3, 3) if cds_str[i:i+3] in ("TGG", "CAG", "CAA", "AAA", "GAA", "TAC", "TAT"))
print(f"Codón elegido en la posición {pos+1}: {cds_str[pos:pos+3]}")

def mutar(s, i):
    # sustitución que genera un paro si es posible, si no, un cambio cualquiera
    for b in "TAGC":
        m = s[:i] + b + s[i+1:]
        if Seq(m[i - i % 3: i - i % 3 + 3]).translate() == "*":
            return m
    return s[:i] + ("A" if s[i] != "A" else "G") + s[i+1:]

mut = mutar(cds_str, pos)
p_wt, p_mut = Seq(cds_str).translate(to_stop=True), Seq(mut).translate(to_stop=True)
N = 10   # ARNm transcritos a partir del gen
error_adn = [p_mut] * N                     # todos los transcritos heredan el error
error_arn = [p_mut] + [p_wt] * (N - 1)      # solo un transcrito defectuoso
print(f"Proteína normal: {len(p_wt)} aa | mutante: {len(p_mut)} aa")
print(f"Error en replicación  -> {sum(p != p_wt for p in error_adn)}/{N} proteínas afectadas (y para siempre)")
print(f"Error en transcripción -> {sum(p != p_wt for p in error_arn)}/{N} proteínas afectadas (solo ese ARNm)")

---
## Entregables
- `dogma_pipeline.py`: generado por la celda `%%writefile`, listo para el repositorio (`python dogma_pipeline.py archivo.fasta --salida prefijo`).
- Ficheros generados: `ej2.fasta`, `INS_cds.fasta`, `INS_resultados.fasta`, `ej2_resultados.fasta`.
- Para el informe (PDF/README.md, máx. 4 páginas) se pueden reutilizar directamente las celdas Markdown de este notebook.